## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [ ]:
H0: mu = 19 -> gate 30
Ha: mu = 20 -> gate 40

Маємо правосторонній тест

2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

In [2]:
df = pd.read_csv(r'C:\Users\mazur\data_analysis\data\cookie_cats.csv')

In [3]:
print(df)


        userid  version  sum_gamerounds  retention_1  retention_7
0          116  gate_30               3        False        False
1          337  gate_30              38         True        False
2          377  gate_40             165         True        False
3          483  gate_40               1        False        False
4          488  gate_40             179         True         True
...        ...      ...             ...          ...          ...
90184  9999441  gate_40              97         True        False
90185  9999479  gate_40              30        False        False
90186  9999710  gate_30              28         True        False
90187  9999768  gate_40              51         True        False
90188  9999861  gate_40              16        False        False

[90189 rows x 5 columns]


In [ ]:
#retention after 7 days 
avg_by_groups = df.groupby('version')['retention_7'].mean()
print(avg_by_groups)

version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64


In [ ]:
# Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?
H0: gate_30 = gate_40
Ha: gate_30 > gate_40

retentio with gate_30 19%
retentio with gate_40 18%

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [14]:
#поділили на групи
control = df[df['version'] == 'gate_30']
treatment = df[df['version'] == 'gate_40']

#успіх: к-ість користувачів які повернулися в гру через 7 днів
success = np.array([
    control['retention_7'].sum(),
    treatment['retention_7'].sum()
])

#заг к-сть користувачів 
nobs = np.array([
    len(control),
    len(treatment)
])

print('успіх: к-ість користувачів які повернулися в гру через 7 днів, success = ', success)
print('заг к-сть користувачів , nobs = ', nobs)

успіх: к-ість користувачів які повернулися в гру через 7 днів, success =  [8502 8279]
заг к-сть користувачів , nobs =  [44700 45489]


In [20]:
#z statistic, p-value: 
stat, pval = proportions_ztest(success, nobs)

print('stat = ', stat)
print('pval = ', pval)

#Довірчий інтервал 95% для групи control: [..., ...]
#Довірчий інтервал 95% для групи treatment: [..., ...]
alpha = 0.05

confint_a = proportion_confint(success[0], nobs[0], alpha=0.05, method='normal')
confint_b = proportion_confint(success[1], nobs[1], alpha=0.05, method='normal')


print('Довірчий інтервал 95% для групи control:', confint_a)
print('Довірчий інтервал 95% для групи treatment: ', confint_b)

stat =  3.164358912748191
pval =  0.001554249975614329
Довірчий інтервал 95% для групи control: (0.18656311652199903, 0.19383956804175934)
Довірчий інтервал 95% для групи treatment:  (0.17845430073314686, 0.18554578720019968)


In [ ]:
      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   

pval =  0.0015 < 0.05 -> різниця між групами є статистично значуща, нульову гіпотезу H0: gate_30 = gate_40 відхиляємо.
Зізниця між групами суттєва.


      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  

Довірчий інтервал 95% для групи control: (18,65%, 19,38%)
Довірчий інтервал 95% для групи treatment:  (17,84%, 18,55%)
Довірчі інтервали 95% для групи control і treatment не перетинаються, що також свідчить про те що Н0 відхиляємо

Retention при gate 30 є значно кращим, якщо перенесемо gate на 40 то можемо втарити користувачів.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


In [23]:

table = [[success[0], (nobs[0]-success[0])],
         [success[1], (nobs[1]-success[1])],
        ]

table

[[np.int64(8502), np.int64(36198)], [np.int64(8279), np.int64(37210)]]

In [25]:
a = pd.crosstab(df['version'], df['retention_7'])
a

retention_7,False,True
version,,
gate_30,36198,8502
gate_40,37210,8279


In [ ]:
#Гіпотези
Н0: версія гри і ретеншин незалежні

In [24]:
import scipy.stats as stats 

chi2, p, dof, expected = stats.chi2_contingency(table)

print(f"χ² = {chi2:.3f}")
print(f"p-value = {p:.5f}")
print(f"Ступені свободи = {dof}")
print("Очікувані частоти:\n", expected)

χ² = 9.959
p-value = 0.00160
Ступені свободи = 1
Очікувані частоти:
 [[ 8317.09742873 36382.90257127]
 [ 8463.90257127 37025.09742873]]


In [ ]:
p-value = 0.00160 < 0.05 -> ми отримали значущий результат і відхиляємо Н0 (тобто події всетаки залежні)